In [ ]:
#Used to test that document parser py is working properly. 

from document_parser import DocumentParser

out_dir = r"datasets\processed"
dp = DocumentParser(out_dir=out_dir)

file_path = r"datasets/raw/COF_HY24_IP.pdf"

dp.ingest_document(file_path=file_path)

c:\Users\chuak\OneDrive\Documents\Projects\LLM-based-information-extraction-for-financial-documents\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing file: COF_HY24_IP.pdf


The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[INFO] 2026-02-08 10:55:38,051 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-08 10:55:38,057 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-02-08 10:55:38,069 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\chuak\OneDrive\Documents\Projects\LLM-based-information-extraction-for-financial-documents\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-08 10:55:38,071 [RapidOCR] main.py:50: Using C:\Users\chuak\OneDrive\Documents\Projects\LLM-based-information-extraction-for-financial-documents\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-08 10:55:38,537 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-08 10:55:38,537 [Rapi

In [78]:
from langchain_docling import DoclingLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

file_path = r"data/raw/COF_HY24_IP.pdf"

loader = DoclingLoader(file_path=file_path)
docs = loader.load()

#Create chunker 
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", "!", "?", "|"]  # Respect tables/MD
)
chunks = splitter.split_documents(docs)  # Now ready for FAISS



[INFO] 2026-02-08 16:12:57,981 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-08 16:12:57,981 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-02-08 16:12:57,983 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\chuak\OneDrive\Documents\Projects\LLM-based-information-extraction-for-financial-documents\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-08 16:12:57,983 [RapidOCR] main.py:50: Using C:\Users\chuak\OneDrive\Documents\Projects\LLM-based-information-extraction-for-financial-documents\.venv\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_infer.pth
[INFO] 2026-02-08 16:12:58,181 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-02-08 16:12:58,181 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-02-08 16:12:58,181 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\chuak\OneDrive\Documents\Projects\LLM-based-information-extraction-for-financial-

In [12]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [ ]:
# 3. Feed to FAISS - embeds & indexes
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)

# 4. Query - gets chunks with source/page... Essentially, similarity search takes the query and find docs that match the query. 
results = vectorstore.similarity_search("What is Centuria's Office cap rate?", k=3)


for doc in results:
    print(f"Page {doc.metadata.get('page')}: {doc.metadata['source']}")
    print(doc.page_content[:200] + "...")

Page None: data/raw/COF_HY24_IP.pdf
Centuria Office REIT (COF)
Australia's largest ASX-listed pure play office REIT . Overseen by an active management team with deep real estate expertise. Strongly supported by Centuria Capital Group...
Page None: data/raw/COF_HY24_IP.pdf
Appendix E: COF leasing history
Centuria continued to deliver a strong track record of maintaining high occupancy...
Page None: data/raw/COF_HY24_IP.pdf
Disclaimer
This presentation has been prepared by Centuria Property Funds Limited (ABN 11 086 553 639, AFSL 231 149) (CPFL) as responsible entity of Centuria Office    T (A               ) ('COF' or t...


In [24]:
for doc in results:
    print(doc.metadata.get('dl_meta').get('doc_items')[0].get('prov')[0].get('page_no'))

5
29
33


In [59]:
from typing import List
from langchain_core.documents import Document

# function takes in list of documents as created from loader.load(). Creates embedding out of these documents. Converts docs into an index using FAISS. 
def build_faiss_index(docs: List[Document]) -> FAISS:
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    # LangChain FAISS wrapper builds the index + stores docs/metadata
    return FAISS.from_documents(docs, embeddings)

# ---------- 4) Retrieve top-k chunks for extraction ----------
def retrieve_context(vs: FAISS, query: str, k: int = 12) -> List[Document]:
    # You can also use similarity_search_with_score for debugging thresholds
    return vs.similarity_search(query, k=k)

In [60]:
#Create a new FAISS vector store using documents that have been created using loader.load()
vs = build_faiss_index(results)

query = "What is Centuria Office REIT's cap rate?"

retrieved_docs = retrieve_context(vs = vs, query = query)



In [61]:
retrieved_docs[0].metadata.get('dl_meta').get('doc_items')[0].get('prov')[0].get('page_no')

5

In [62]:
retrieved_docs[0].metadata

{'source': 'data/raw/COF_HY24_IP.pdf',
 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta',
  'version': '1.0.0',
  'doc_items': [{'self_ref': '#/texts/51',
    'parent': {'$ref': '#/body'},
    'children': [],
    'content_layer': 'body',
    'label': 'text',
    'prov': [{'page_no': 5,
      'bbox': {'l': 561.29,
       't': 408.989,
       'r': 915.195,
       'b': 337.044,
       'coord_origin': 'BOTTOMLEFT'},
      'charspan': [0, 170]}]}],
  'headings': ['Centuria Office REIT (COF)'],
  'origin': {'mimetype': 'application/pdf',
   'binary_hash': 2594346747007363658,
   'filename': 'COF_HY24_IP.pdf'}}}

In [73]:
from pydantic import BaseModel
from pydantic import Field
from typing import Optional

class Metric(BaseModel):
    metric_name_doc: Optional[str] = Field(description="Exact name of metric being queried in document")
    metric_val:Optional[float] = Field(description="Value of metric being extracted")
    metric_period:Optional[str] = Field(description="Financial Period of Metric period extracted. Eg: FYXX, HYXX")
    # metric_page_source:Optional[str] = Field(description="Page number of value of metric extracted")


In [74]:
from openai import OpenAI


# ---------- 5) Structured extraction with OpenAI (single call) ----------
def extract_metrics_structured(retrieved_docs: List[Document], user_goal: str):
    client = OpenAI()

    # Put chunk ids next to text so the model can cite them in Evidence.chunk_id
    context_blocks = []
    for d in retrieved_docs:
        #chunk id, source of document, page of document
        cid = d.metadata.get('dl_meta').get('doc_items')[0].get('self_ref')
        src = d.metadata.get("source")
        page = d.metadata.get('dl_meta').get('doc_items')[0].get('prov')[0].get('page_no')

        header = f"[chunk_id={cid} source={src} page={page}]"
        context_blocks.append(f"{header}\n{d.page_content}")
        print(context_blocks)

    sample_messages = [
    {"role":"system","content":"You are a financial extractor"},
    {f"role":"user","content":f"Context:\n{context_blocks}"},
    {f"role":"user","content":"What is Centuria Office REIT's cap rate?"}
    ]


    model_name = "gpt-4o-mini"

    #parse is for structured output
    completion = client.chat.completions.parse(
        model = model_name,
        messages = sample_messages,
        response_format=Metric
    )

    return completion

In [75]:
completion = extract_metrics_structured(retrieved_docs=retrieved_docs,user_goal=None)

["[chunk_id=#/texts/51 source=data/raw/COF_HY24_IP.pdf page=5]\nCenturia Office REIT (COF)\nAustralia's largest ASX-listed pure play office REIT . Overseen by an active management team with deep real estate expertise. Strongly supported by Centuria Capital Group"]
["[chunk_id=#/texts/51 source=data/raw/COF_HY24_IP.pdf page=5]\nCenturia Office REIT (COF)\nAustralia's largest ASX-listed pure play office REIT . Overseen by an active management team with deep real estate expertise. Strongly supported by Centuria Capital Group", '[chunk_id=#/texts/479 source=data/raw/COF_HY24_IP.pdf page=29]\nAppendix E: COF leasing history\nCenturia continued to deliver a strong track record of maintaining high occupancy']
["[chunk_id=#/texts/51 source=data/raw/COF_HY24_IP.pdf page=5]\nCenturia Office REIT (COF)\nAustralia's largest ASX-listed pure play office REIT . Overseen by an active management team with deep real estate expertise. Strongly supported by Centuria Capital Group", '[chunk_id=#/texts/479 

In [76]:
completion.choices[0].message.content

'{"metric_name_doc":"Cap Rate","metric_val":null,"metric_period":null}'